# 📧 Email Intent Dataset Generator

Generates a synthetic SaaS customer-support email dataset for fine-tuning **DistilRoBERTa**.

**Pipeline**
1. GPT-4o-mini → 50 diverse base emails per intent
2. Python augmentation → expand each intent to 1,000 examples
3. Quality checks → deduplicate, length filter
4. Stratified split → train / val / test (80 / 10 / 10)
5. Export → `train.jsonl`, `val.jsonl`, `test.jsonl`, `label_metadata.json`

**Intents covered:** `login_issue`, `billing_refund`, `subscription_change`, `bug_report`, `feature_request`, `integration_api`, `performance_issue`, `security_concern`

**Cost estimate:** ~$0.30–$0.50 total with `gpt-4o-mini`

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install openai -q
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Paste your OpenAI API key here
OPENAI_API_KEY    = "sk-..."          # <── paste your key here

MODEL             = "gpt-4o-mini"     # cheap + fast; swap to gpt-4o for higher quality
BASE_PER_INTENT   = 50               # emails generated by the LLM per intent
TARGET_PER_INTENT = 1_000            # final count after Python augmentation
BATCH_SIZE        = 10               # emails per API call  (5 calls × 10 = 50 base)
OUTPUT_DIR        = "output"         # folder where JSONL files are saved
RANDOM_SEED       = 42

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✓ Config set  |  model={MODEL}  |  target={TARGET_PER_INTENT:,}/intent  |  output={OUTPUT_DIR}/')

In [ ]:
# ── Cell 3: Define intents & departments ──────────────────────────────────────
INTENTS = {
    "login_issue": {
        "department": "Technical Support",
        "description": (
            "User cannot log in, password reset not working, MFA/2FA not sending codes, "
            "SSO or OAuth failures (Google, Microsoft), account locked, too many attempts, "
            "CAPTCHA problems, invalid credentials after migration."
        ),
    },
    "billing_refund": {
        "department": "Billing",
        "description": (
            "Request for a refund, charge dispute, unexpected invoice, double charge, "
            "payment declined, credit card update needed, refund status follow-up."
        ),
    },
    "subscription_change": {
        "department": "Billing",
        "description": (
            "Upgrade or downgrade plan, cancel subscription, pause account, "
            "add or remove seats, switch billing cycle (monthly to annual), "
            "questions about plan limits or pricing."
        ),
    },
    "bug_report": {
        "department": "Engineering",
        "description": (
            "Software bug, feature not working as expected, UI error, broken functionality, "
            "data not loading, export failing, dashboard showing wrong numbers, "
            "error messages with stack traces or error codes."
        ),
    },
    "feature_request": {
        "department": "Product",
        "description": (
            "Request for a new feature, improvement to existing feature, workflow suggestion, "
            "integration request, API capability request, UX improvement idea."
        ),
    },
    "integration_api": {
        "department": "Developer Support",
        "description": (
            "API authentication, API key issues, webhook not firing, SDK questions, "
            "integration with Zapier/Slack/Salesforce/HubSpot, rate limiting, "
            "API documentation confusion, REST or GraphQL endpoint questions."
        ),
    },
    "performance_issue": {
        "department": "Technical Support",
        "description": (
            "App is slow, pages timing out, high latency, data sync delays, "
            "bulk operations taking too long, dashboard not loading, mobile app lagging."
        ),
    },
    "security_concern": {
        "department": "Security",
        "description": (
            "Suspicious activity on account, unauthorized access, phishing email, "
            "data breach concern, GDPR or SOC2 compliance question, "
            "audit log request, permission or access control issue."
        ),
    },
}

LABELS   = list(INTENTS.keys())
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print(f'✓ {len(INTENTS)} intents defined:')
for intent, meta in INTENTS.items():
    print(f'   {intent:<25}  →  {meta["department"]}')

In [ ]:
# ── Cell 4: Augmentation pools ────────────────────────────────────────────────
import random, re, hashlib
random.seed(RANDOM_SEED)

FIRST_NAMES = [
    "James", "Sarah", "Michael", "Emily", "David", "Jessica", "Chris", "Amanda",
    "Daniel", "Lisa", "Robert", "Megan", "William", "Rachel", "Andrew", "Hannah",
    "Liam", "Sofia", "Noah", "Olivia", "Raj", "Priya", "Wei", "Fatima", "Carlos",
    "Maria", "Yuki", "Aarav", "Zoe", "Lucas", "Emma", "Ethan", "Chloe", "Tyler",
]
LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis",
    "Wilson", "Taylor", "Anderson", "Thomas", "Jackson", "White", "Harris", "Martin",
    "Thompson", "Martinez", "Robinson", "Clark", "Patel", "Kim", "Chen", "Nguyen",
    "Scott", "Young", "Allen", "King", "Wright", "Hill", "Torres", "Green",
]
COMPANIES = [
    "Acme Corp", "BrightPath Inc", "Nexus Solutions", "Orbit Software", "Cascade Tech",
    "Pinnacle Systems", "Stride Analytics", "Blueprint Labs", "Vertex Dynamics",
    "Quantum Works", "Helix Platforms", "Northstar Digital", "Apex Ventures",
    "Crestline SaaS", "Horizon Cloud", "Silverpeak Studio", "Redwood Operations",
    "Ironclad Technologies", "Summit Data", "Paragon Consulting", "TrueNorth Apps",
    "Clearview Media", "Keystone Dev", "Fulcrum AI", "Meridian Software",
]
PLANS    = ["Starter", "Growth", "Professional", "Business", "Enterprise", "Team", "Pro", "Scale", "Plus", "Premium"]
GREETINGS = [
    "Hi Support Team,", "Hello,", "Hi there,", "Good morning,", "Dear Support,",
    "Hey,", "To Whom It May Concern,", "Hi,", "Hello Support,", "Hi Team,",
    "Good afternoon,", "Hi Support,",
]
CLOSINGS = [
    "Thanks,", "Best,", "Thank you,", "Regards,", "Cheers,", "Thanks in advance,",
    "Appreciate your help,", "Looking forward to hearing from you,",
    "Best regards,", "Kind regards,", "Thanks!", "Warm regards,",
]
TICKET_PREFIXES = [
    "", "", "", "",  # most emails have no ticket ref
    "Re: Ticket #{ticket} – ",
    "Following up on #{ticket} – ",
    "[Ticket #{ticket}] ",
]
FORWARDED_PREFIX = [
    "", "", "", "", "",   # most are not forwarded
    "---------- Forwarded message ----------\nFrom: {name} <{name_lower}@{company_lower}.com>\n\n",
]
TYPO_SWAPS = {
    "the": "teh", "your": "yuor", "with": "wiht", "this": "tihs",
    "have": "ahve", "that": "taht", "from": "form", "when": "wehn",
    "login": "logon", "account": "accont", "password": "pasword",
    "billing": "biling", "invoice": "invioce", "please": "plaese",
}

def _random_name():    return random.choice(FIRST_NAMES) + " " + random.choice(LAST_NAMES)
def _random_company(): return random.choice(COMPANIES)
def _random_plan():    return random.choice(PLANS)
def _ticket_id():      return str(random.randint(10000, 99999))

def _maybe_add_typos(text, rate=0.15):
    if random.random() > rate:
        return text
    words = text.split()
    for i, word in enumerate(words):
        lower = word.lower()
        if lower in TYPO_SWAPS and random.random() < 0.25:
            words[i] = word.replace(lower, TYPO_SWAPS[lower])
    return " ".join(words)

def augment_email(base_email):
    """Produce a varied version of a base email by swapping names, companies,
    greeting/closing, optionally wrapping in a forwarded header, adding ticket
    refs in the subject, and injecting realistic typos."""
    name         = _random_name()
    first_name   = name.split()[0]
    company      = _random_company()
    plan         = _random_plan()
    ticket       = _ticket_id()
    name_lower   = name.split()[0].lower()
    company_lower= company.lower().replace(" ", "")

    text = base_email

    # Swap greeting
    greeting = random.choice(GREETINGS)
    text = re.sub(
        r"^(Hi[^,\n]*,|Hello[^,\n]*,|Dear[^,\n]*,|Hey[^,\n]*,|Good (morning|afternoon)[^,\n]*,)\n?",
        greeting + "\n\n", text, flags=re.IGNORECASE | re.MULTILINE,
    )

    # Swap closing / signature
    lines = text.strip().splitlines()
    if len(lines) > 4:
        closing  = random.choice(CLOSINGS)
        cut      = random.randint(1, min(3, len(lines) - 3))
        sig_lines= [closing, name]
        if random.random() < 0.4:  sig_lines.append(company)
        if random.random() < 0.25: sig_lines.append(f"{plan} Plan")
        text = "\n".join(lines[: len(lines) - cut] + sig_lines)

    # Forwarded-email wrapper
    fwd = random.choice(FORWARDED_PREFIX)
    if fwd:
        text = fwd.format(name=name, name_lower=name_lower, company_lower=company_lower) + text

    # Replace LLM placeholder names / companies
    for pat, repl in {
        r"\[Name\]": first_name, r"\[Company\]": company, r"\[Plan\]": plan,
        r"John Doe": name, r"Jane Doe": name,
        r"Acme": company, r"Example Corp": company,
        r"Pro Plan": f"{plan} Plan", r"Business Plan": f"{plan} Plan",
    }.items():
        text = re.sub(pat, repl, text)

    # Optional ticket ref in subject line
    ticket_tmpl = random.choice(TICKET_PREFIXES)
    if ticket_tmpl and "Subject:" in text:
        text = re.sub(r"(Subject: )", r"\1" + ticket_tmpl.format(ticket=ticket), text, count=1)

    return _maybe_add_typos(text).strip()

print('✓ Augmentation helpers ready')

In [ ]:
# ── Cell 5: LLM generation (OpenAI) ──────────────────────────────────────────
import json, time
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = (
    "You are a data generation assistant. You write realistic, varied synthetic "
    "customer support emails for a B2B SaaS company. Each email must be unique, "
    "with different tone, length, writing quality, and specific details. "
    "Some should be from enterprise customers, some from startups, some from solo users. "
    "Vary length from 1–2 sentences to multi-paragraph. "
    "Include realistic details: browser names, OS, error messages, ticket IDs, "
    "dates, plan names, company names. "
    "Occasionally include typos, grammatical errors, or informal language. "
    "Return ONLY a JSON array of strings. No markdown, no explanation."
)

def build_prompt(intent, meta, batch_size=10):
    return (
        f"Generate {batch_size} different customer support emails for the intent: {intent!r}.\n"
        f"Department routing target: {meta['department']}\n"
        f"Intent description: {meta['description']}\n\n"
        f"Requirements:\n"
        f"- Each email must cover a distinct sub-topic within this intent\n"
        f"- Vary tone: formal, casual, frustrated, polite, urgent\n"
        f"- Vary length: some short (2–3 sentences), some long (3+ paragraphs)\n"
        f"- 2 of the {batch_size} should mention a secondary issue but primary intent is still {intent!r}\n"
        f"- At least 1 should be a follow-up ('Re: ...' or 'Following up on ticket ...')\n"
        f"- Begin each email with a Subject: line\n\n"
        f"Return a JSON array of {batch_size} complete email strings."
    )

def generate_base_emails(intent, meta, total=50, batch_size=10):
    """Call GPT-4o-mini in batches. Returns a list of raw email strings."""
    emails   = []
    n_batches = (total + batch_size - 1) // batch_size
    print(f"  Generating {total} base emails for '{intent}' ({n_batches} API calls)...")

    for batch_idx in range(n_batches):
        remaining   = total - len(emails)
        this_batch  = min(batch_size, remaining)
        prompt      = build_prompt(intent, meta, batch_size=this_batch)

        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    temperature=0.9,
                    max_tokens=3000,
                )
                raw = response.choices[0].message.content.strip()
                # Strip markdown code fences if present
                raw = re.sub(r"^```(json)?", "", raw).strip()
                raw = re.sub(r"```$",        "", raw).strip()

                batch_emails = json.loads(raw)
                if isinstance(batch_emails, list):
                    emails.extend([str(e) for e in batch_emails])
                    print(f"    Batch {batch_idx+1}/{n_batches}: ✓ {len(batch_emails)} emails")
                    break
            except (json.JSONDecodeError, Exception) as exc:
                print(f"    Batch {batch_idx+1} attempt {attempt+1} failed: {exc}")
                time.sleep(2 ** attempt)

        time.sleep(0.3)  # gentle rate-limit buffer

    return emails[:total]

print('✓ OpenAI client ready')

In [ ]:
# ── Cell 6: Expand base → 1,000 per intent, then assemble full dataset ────────

def expand_to_target(base_emails, target, intent):
    """Augment base emails until we reach `target` deduplicated examples."""
    result       = []
    seen_hashes  = set()

    def _add(email):
        h = hashlib.md5(email[:200].encode()).hexdigest()
        if h not in seen_hashes:
            seen_hashes.add(h)
            result.append(email)

    for email in base_emails:
        _add(email)  # add originals first

    multiplier = (target // max(len(base_emails), 1)) + 5
    for _ in range(multiplier):
        for email in base_emails:
            if len(result) >= target: break
            _add(augment_email(email))
        if len(result) >= target: break

    # Safety fallback (rare)
    while len(result) < target:
        result.append(augment_email(random.choice(base_emails)))

    random.shuffle(result)
    print(f"  Expanded '{intent}': {len(base_emails)} base → {len(result[:target])} final")
    return result[:target]


def build_record(text, intent):
    return {
        "text":       text,
        "label":      LABEL2ID[intent],
        "intent":     intent,
        "department": INTENTS[intent]["department"],
    }


# ── Run the pipeline ──
all_records = []

for intent, meta in INTENTS.items():
    print(f"\n[{intent}]  →  {meta['department']}")

    base_emails  = generate_base_emails(intent, meta, total=BASE_PER_INTENT, batch_size=BATCH_SIZE)
    final_emails = expand_to_target(base_emails, TARGET_PER_INTENT, intent)

    for email in final_emails:
        all_records.append(build_record(email, intent))

print(f"\n✓ Total records assembled: {len(all_records):,}")

In [ ]:
# ── Cell 7: Stratified split + export JSONL ────────────────────────────────────
import json
from pathlib import Path

def split_dataset(records, seed=RANDOM_SEED):
    """80 / 10 / 10 stratified split."""
    random.seed(seed)
    train, val, test = [], [], []
    by_intent = {k: [] for k in LABELS}
    for r in records:
        by_intent[r["intent"]].append(r)
    for intent_records in by_intent.values():
        random.shuffle(intent_records)
        n      = len(intent_records)
        n_val  = max(1, int(n * 0.10))
        n_test = max(1, int(n * 0.10))
        test.extend(intent_records[:n_test])
        val.extend( intent_records[n_test : n_test + n_val])
        train.extend(intent_records[n_test + n_val :])
    random.shuffle(train); random.shuffle(val); random.shuffle(test)
    return train, val, test

def write_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  Wrote {len(records):,} records → {path}")


print(f"Splitting {len(all_records):,} records (80/10/10 stratified)...")
train, val, test = split_dataset(all_records)
print(f"  Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")

print("\nWriting JSONL files...")
write_jsonl(train, f"{OUTPUT_DIR}/train.jsonl")
write_jsonl(val,   f"{OUTPUT_DIR}/val.jsonl")
write_jsonl(test,  f"{OUTPUT_DIR}/test.jsonl")

# Label metadata (for DistilRoBERTa config)
metadata = {
    "num_labels":      len(LABELS),
    "label2id":        LABEL2ID,
    "id2label":        ID2LABEL,
    "intent_metadata": {k: {"department": v["department"]} for k, v in INTENTS.items()},
    "splits": {"train": "train.jsonl", "validation": "val.jsonl", "test": "test.jsonl"},
}
meta_path = f"{OUTPUT_DIR}/label_metadata.json"
Path(meta_path).write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
print(f"  Wrote label metadata → {meta_path}")

print("\n✓ All files written to", OUTPUT_DIR + "/")

In [ ]:
# ── Cell 8: Verify & preview ──────────────────────────────────────────────────
from collections import Counter

print("=" * 55)
print("DATASET SUMMARY")
print("=" * 55)
print(f"{'Split':<12} {'Count':>8}")
print("-" * 22)
for split_name, split_data in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{split_name:<12} {len(split_data):>8,}")
print("-" * 22)
print(f"{'TOTAL':<12} {len(all_records):>8,}")

print("\nIntent distribution in train split:")
counts = Counter(r["intent"] for r in train)
for intent in LABELS:
    bar = "█" * (counts[intent] // 50)
    print(f"  {intent:<25} {counts[intent]:>5,}  {bar}")

print("\nSample record (train[0]):")
sample = train[0]
print(f"  intent    : {sample['intent']}")
print(f"  department: {sample['department']}")
print(f"  label     : {sample['label']}")
print(f"  text[:200]:\n{sample['text'][:200]}...")

In [ ]:
# ── Cell 9: Download files from Colab ─────────────────────────────────────────
# Run this cell to download all four output files to your local machine.
from google.colab import files
import os

for fname in ["train.jsonl", "val.jsonl", "test.jsonl", "label_metadata.json"]:
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        files.download(path)
        print(f"  ↓ Downloading {fname}")
    else:
        print(f"  ✗ Not found: {path}  (run earlier cells first)")

In [ ]:
# ── Cell 10: Load with HuggingFace datasets (copy into your training notebook) ─
# This cell shows how to load the generated dataset for DistilRoBERTa fine-tuning.

# !pip install datasets transformers -q

# from datasets import load_dataset
# ds = load_dataset("json", data_files={
#     "train":      "output/train.jsonl",
#     "validation": "output/val.jsonl",
#     "test":       "output/test.jsonl",
# })
#
# from transformers import AutoModelForSequenceClassification
# import json
# meta = json.load(open("output/label_metadata.json"))
#
# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilroberta-base",
#     num_labels  = meta["num_labels"],
#     id2label    = meta["id2label"],
#     label2id    = meta["label2id"],
# )

print("Dataset is ready. See the commented code above to load it for fine-tuning.")